In [6]:
import torch

In [7]:
# 단일 변수 합성함수 y = sin(x²)의 연쇄 법칙을 계산한다.
x = torch.tensor(2.0, dtype=torch.float64)

# 순전파: 입력 x에서 중간값 u를 거쳐 출력 y를 계산한다.
u = x.pow(2)
y = torch.sin(u)

dy_du = torch.cos(u)
du_dx = 2 * x

# total 미분
dy_dx = dy_du * du_dx

print("x:", x.item())
print("u = x²:", u.item())
print("y = sin(u):", y.item())
print("dy/dx:", dy_dx.item())

x: 2.0
u = x²: 4.0
y = sin(u): -0.7568024953079282
dy/dx: -2.6145744834544478


In [8]:
def forward_function(x):
    """x -> u -> y의 순전파 계산을 수행한다."""
    x1 = x[0]
    x2 = x[1]
    
    # 두 입력으로부터 중간 변수 u₁, u₂를 만든다.
    u1 = x1 + x2
    u2 = x1 * x2
    u = torch.stack((u1, u2))

    # 중간 변수들을 이용해 최종 scalar 출력 y를 계산한다.
    y = u1**2 + 3 * u2
    
    return u, y

In [9]:
x = torch.tensor([1.0, 2.0])

# 순전파로 중간값 u와 최종 출력 y를 계산한다.
u, y = forward_function(x)

# y = u₁² + 3u₂를 u₁, u₂에 대해 편미분한다.
gradient_u = torch.tensor([
    2 * u[0],
    3.0,
])

# A[i, j]에는 ∂uⱼ/∂xᵢ가 들어간다.
A = torch.tensor([
    [1.0, x[1]],  # ∂u₁/∂x₁, ∂u₂/∂x₁
    [1.0, x[0]],  # ∂u₁/∂x₂, ∂u₂/∂x₂
])

# 중간 변수로 들어온 gradient를 입력 x까지 전달한다.
gradient_x = A @ gradient_u

print("u:", u)
print("y:", y)
print("Gradient with respect to u:", gradient_u)
print("Gradient with respect to x:", gradient_x)

u: tensor([3., 2.])
y: tensor(15.)
Gradient with respect to u: tensor([6., 3.])
Gradient with respect to x: tensor([12.,  9.])


In [11]:
def scalar_output(x):
    """중간 변수를 숨기고 x에서 y까지 전체 함수를 계산한다."""
    _, y = forward_function(x)
    return y


# 중앙 차분으로 직접 구한 gradient와 연쇄 법칙 결과를 비교한다.
h = 1e-4
numerical_gradient = torch.empty_like(x)

for index in range(x.numel()):
    # 한 번에 하나의 입력 변수만 h만큼 변화시킨다.
    perturbation = torch.zeros_like(x)
    perturbation[index] = h

    numerical_gradient[index] = (
        scalar_output(x + perturbation)
        - scalar_output(x - perturbation)
    ) / (2 * h)

print("Chain rule gradient:", gradient_x)
print("Numerical gradient:", numerical_gradient)


Chain rule gradient: tensor([12.,  9.])
Numerical gradient: tensor([12.0068,  8.9979])
